
## ETL/Silver/01 - Bronze -> Silver
## Salida: pf.silver.modelos  y  pf.silver.model_tag
## spark.sql para transformaciones + Dataframe API para dedup/escritura.

In [0]:
%py

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG = get_param("catalog", "pf")
BRONZE  = f"{CATALOG}.bronze.models_raw"
S_MODELO = f"{CATALOG}.silver.modelos"
S_TAG    = f"{CATALOG}.silver.model_tag"

from pyspark.sql import functions as F
from pyspark.sql.functions import explode

In [0]:
%py
df_sil = spark.sql(f"""
    SELECT
        id AS model_id,
        SPLIT(id, '/')[1] AS nombre,
        SPLIT(id, '/')[0] AS org_id,
        CAST(likes AS BIGINT) AS likes,
        CAST(downloads AS BIGINT) AS downloads,
        CAST(private AS BOOLEAN) AS private,
        NULLIF(pipeline_tag, '') AS pipeline_tag,
        NULLIF(library_name, '') AS library_name,
        NULLIF(try_element_at(filter(from_json(tags, 'array<string>'), t -> t LIKE 'license:%'), 1), '') AS license_tag,
        CAST(createdAt AS TIMESTAMP) AS createdAt,
        CAST(lastModified AS TIMESTAMP) AS lastModified,
        ingestion_date,
        (payload_json IS NOT NULL) AS has_metadata,
        _rescued_data
    FROM {BRONZE}
""")

In [0]:
%py

n_antes = df_sil.count()
df_sil = df_sil.dropDuplicates(["model_id", "ingestion_date"])
n_despues = df_sil.count()
print(f"Dedup: {n_antes} -> {n_despues} filas ({(n_antes - n_despues)} duplicados eliminados)")

In [0]:
%py

fechas = [r[0].isoformat() for r in
          spark.sql(f"SELECT DISTINCT ingestion_date FROM {BRONZE}").collect()]
replace_where = "ingestion_date IN (" + ",".join(f"date'{d}'" for d in fechas) + ")"

(df_sil.write
   .mode("overwrite")
   .option("replaceWhere", replace_where)
   .partitionBy("ingestion_date")
   .format("delta")
   .saveAsTable(S_MODELO))

print("Silver.modelos actualizada")

In [0]:
%py

df_tag = spark.sql(f"""
    SELECT
        id AS model_id,
        explode(from_json(tags, 'array<string>')) AS tag,
        ingestion_date
    FROM {BRONZE}
    WHERE COALESCE(tags, '[]') != '[]'
""")

df_tag = df_tag.select(
    "model_id",
    "tag",
    F.col("tag").startswith("license:").alias("es_license"),
    F.col("tag").startswith("dataset:").alias("es_dataset"),
    F.col("tag").startswith("arxiv:").alias("es_arxiv"),
    "ingestion_date",
)

(df_tag.write
   .mode("overwrite")
   .option("replaceWhere", replace_where)
   .partitionBy("ingestion_date")
   .format("delta")
   .saveAsTable(S_TAG))

print(f"Silver.model_tag: {df_tag.count()} filas")

In [0]:
%py

spark.sql(f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN has_metadata THEN 1 ELSE 0 END) AS con_payload,
        SUM(CASE WHEN license_tag IS NULL THEN 1 ELSE 0 END) AS sin_licencia,
        COUNT(DISTINCT model_id) AS modelos_unicos
    FROM {S_MODELO}
""").show()

In [0]:
%sql

SELECT
    (SELECT COUNT(*) FROM pf.bronze.models_raw) as reg_bronze,
    (SELECT COUNT(*) FROM pf.silver.modelos) as reg_silver;